In [2]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata
import os
import seaborn as sb
import matplotlib
from harmony import harmonize

In [3]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

# Ignore R warning messages
#Note: this can be commented out to get more verbose R output
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

# Automatically convert rpy2 outputs to pandas dataframes
pandas2ri.activate()
anndata2ri.activate()
%load_ext rpy2.ipython

/tmp/ipykernel_2172117/2253401465.py:13: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


# GSE141031##

In [9]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output/'
adata <- readRDS(file.path(outdir,"GSE141031/GSE141031_pre_annt.rds"))
level1_marker <- list(
    # "B cell" = c('Cd79a', 'Cd79b', 'Ms4a1', 'Igkc', 'Cd22', 'Fcer2'),
    'T cell'= c('Cd2', 'Trac', 'Cd69', 'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Eomes', 'Lag3'),
    # 'Natural killer cell'= c('Nkg7', 'Xcl1', 'Ctsw', 'Xcl2', 'Cd160', 'Fcgr3', 'Prf1', 'Gnly'),

    # 'Dendritic cell'= c('Clec10a', 'Fcer1a', 'Cd1d1', 'H2-Ab1', 'H2-Eb1'),
    'Macrophage'= c('C1qa', 'C1qb', 'C1qc', 'Cd74', 'Cxcl8', 'Aif1', 'Cd68', 'Itgam', 'Csf1r', 'H2-Ab1', 'Lgals3', 'Cd163'),
    'Monocyte'= c('Fcn1', 'S100a8', 'S100a9', 'S100a12', 'Vcan', 'Cd52', 'Lyz2', 'Ctss', 'Cd14'),
    # 'Mast cell'= c('Tpsab1', 'Tpsb2', 'Hdc', 'Cma1'),
    'Erythrocyte/Erythroid'= c('Hbb-bs', 'Hba-a1', 'Hba-a2', 'Alas2', 'Ahsp', 'Slc4a1', 'Gypa', 'Klf1', 'Tmcc2'),
    'Neutrophil' = c('Nampt', 'Ifitm2', 'G0s2', 'Cxcl8', 'Neat1', 'Srgn', 'Aqp9', 'Sod2', 'Fcgr3', 'Ivns1abp'),
    'Basophil' = c('Tpsb2', 'Cpa3', 'Slc24a3', 'Fer', 'Kit', 'Hpgds', 'Sytl3', 'Maml3', 'Ell2', 'Ccac200', 'Akap13', 'Areg', 'Rhoh', 'Lrmda', 'Arid1b', 'Irak3', 'Tex14', 'Hpgd', 'Ercc1', 'Ctnnbl1', 'Zbtb20'),

    'Endothelial cell'= c('Pecam1', 'Vwf', 'Fabp4', 'Cldn5', 'Ifi27l2a', 'Ecscr', 'Dysf', 'Cd34', 'Col4a1', 'Col4a2', 'Sparcl1', 'Plvap', 'Mpzl2', 'Sulf1', 'Edn1'),

    'Fibroblast'= c('Lum', 'Dcn', 'Col1a1', 'Col1a2', 'Fbln1', 'Thy1'),
    'Smooth muscle cell'= c('Acta2', 'Myh11', 'Myl9', 'Tpm2', 'Cald1', 'Tagln', 'Tnfrsf11b', 'Lum', 'Apoe', 'Apoc1', 'Agt', 'Notch3', 'Pdgfrb', 'Mfap4'),

    'Pericyte'= c('Tagln', 'Lpp', 'Cald1', 'Tpm2', 'Myl9', 'Acta2', 'Map1b', 'Prkg1', 'Igfbp5', 'Synpo2', 'Eps8', 'Timp3', 'Lmod1', 'C11orf96', 'Inpp4b', 'Notch3', 'Ebf1', 'Steap4', 'mt-Rnr1', 'Crispld2', 'Sox5', 'Ppp1r14a', 'Filip1l', 'Lhfpl6', 'Ptprg')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE141031/GSE141031_pre_annt.csv"),row.names = FALSE, )

List of 10
 $ T cell               : chr [1:10] "Cd2" "Trac" "Cd69" "Cd3d" ...
 $ Macrophage           : chr [1:12] "C1qa" "C1qb" "C1qc" "Cd74" ...
 $ Monocyte             : chr [1:9] "Fcn1" "S100a8" "S100a9" "S100a12" ...
 $ Erythrocyte/Erythroid: chr [1:9] "Hbb-bs" "Hba-a1" "Hba-a2" "Alas2" ...
 $ Neutrophil           : chr [1:10] "Nampt" "Ifitm2" "G0s2" "Cxcl8" ...
 $ Basophil             : chr [1:21] "Tpsb2" "Cpa3" "Slc24a3" "Fer" ...
 $ Endothelial cell     : chr [1:15] "Pecam1" "Vwf" "Fabp4" "Cldn5" ...
 $ Fibroblast           : chr [1:6] "Lum" "Dcn" "Col1a1" "Col1a2" ...
 $ Smooth muscle cell   : chr [1:14] "Acta2" "Myh11" "Myl9" "Tpm2" ...
 $ Pericyte             : chr [1:25] "Tagln" "Lpp" "Cald1" "Tpm2" ...


# GSE233625

In [ ]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output/'
adata <- readRDS(file.path(outdir,"GSE233625/GSE233625_pre_annt.rds"))
level1_marker <- list(
    # "B cell" = c('Cd79a', 'Cd79b', 'Ms4a1', 'Igkc', 'Cd22', 'Fcer2'),
    'T cell'= c('Cd2', 'Trac', 'Cd69', 'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Eomes', 'Lag3'),
    # 'Natural killer cell'= c('Nkg7', 'Xcl1', 'Ctsw', 'Xcl2', 'Cd160', 'Fcgr3', 'Prf1', 'Gnly'),

    # 'Dendritic cell'= c('Clec10a', 'Fcer1a', 'Cd1d1', 'H2-Ab1', 'H2-Eb1'),
    'Macrophage'= c('C1qa', 'C1qb', 'C1qc', 'Cd74', 'Cxcl8', 'Aif1', 'Cd68', 'Itgam', 'Csf1r', 'H2-Ab1', 'Lgals3', 'Cd163'),
    'Monocyte'= c('Fcn1', 'S100a8', 'S100a9', 'S100a12', 'Vcan', 'Cd52', 'Lyz2', 'Ctss', 'Cd14'),
    # 'Mast cell'= c('Tpsab1', 'Tpsb2', 'Hdc', 'Cma1'),
    'Erythrocyte/Erythroid'= c('Hbb-bs', 'Hba-a1', 'Hba-a2', 'Alas2', 'Ahsp', 'Slc4a1', 'Gypa', 'Klf1', 'Tmcc2'),
    'Neutrophil' = c('Nampt', 'Ifitm2', 'G0s2', 'Cxcl8', 'Neat1', 'Srgn', 'Aqp9', 'Sod2', 'Fcgr3', 'Ivns1abp'),
    'Basophil' = c('Tpsb2', 'Cpa3', 'Slc24a3', 'Fer', 'Kit', 'Hpgds', 'Sytl3', 'Maml3', 'Ell2', 'Ccac200', 'Akap13', 'Areg', 'Rhoh', 'Lrmda', 'Arid1b', 'Irak3', 'Tex14', 'Hpgd', 'Ercc1', 'Ctnnbl1', 'Zbtb20'),

    'Endothelial cell'= c('Pecam1', 'Vwf', 'Fabp4', 'Cldn5', 'Ifi27l2a', 'Ecscr', 'Dysf', 'Cd34', 'Col4a1', 'Col4a2', 'Sparcl1', 'Plvap', 'Mpzl2', 'Sulf1', 'Edn1'),

    'Fibroblast'= c('Lum', 'Dcn', 'Col1a1', 'Col1a2', 'Fbln1', 'Thy1'),
    'Smooth muscle cell'= c('Acta2', 'Myh11', 'Myl9', 'Tpm2', 'Cald1', 'Tagln', 'Tnfrsf11b', 'Lum', 'Apoe', 'Apoc1', 'Agt', 'Notch3', 'Pdgfrb', 'Mfap4'),

    'Pericyte'= c('Tagln', 'Lpp', 'Cald1', 'Tpm2', 'Myl9', 'Acta2', 'Map1b', 'Prkg1', 'Igfbp5', 'Synpo2', 'Eps8', 'Timp3', 'Lmod1', 'C11orf96', 'Inpp4b', 'Notch3', 'Ebf1', 'Steap4', 'mt-Rnr1', 'Crispld2', 'Sox5', 'Ppp1r14a', 'Filip1l', 'Lhfpl6', 'Ptprg')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE233625/GSE233625_pre_annt.csv"),row.names = FALSE, )

## new_marker

In [4]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output_new_marker/'
adata <- readRDS(file.path(outdir,"GSE233625/GSE233625_pre_annt.rds"))
level1_marker <- list(
    'B cell' = c('Cd79a', 'Cd79b', 'Ms4a1', 'Ly6d'),
    'T cell' = c('Cd3d', 'Cd3g', 'Trbc2'),
    'Natural killer cell' = c('Nkg7', 'Klrb1c', 'Gzma', 'Gzmb', 'Prf1'),
    'Dendritic cell' = c('Cd209a', 'H2-Ab1', 'H2-Eb1', 'Itgax', 'Flt3'),
    'Macrophage' = c('Cd68', 'Adgre1', 'C1qa', 'C1qb', 'Lyz2'),
    'Monocyte' = c('Lyz2', 'Cd14', 'Ccr2', 'Ly6c2', 'Cx3cr1'),
    'Mast cell' = c('Kit', 'Cpa3', 'Cma1', 'Mcpt4', 'Tpsb2'),
    'Erythrocyte/Erythroid' = c('Hba-a1', 'Hba-a2', 'Hbb-bs', 'Hbb-bt', 'Alas2', 'Klf1'),
    'Neutrophil' = c('S100a8', 'S100a9', 'Ly6g', 'Cxcr2', 'Retnlg'),
    'Basophil' = c('Mcpt8', 'Cd200r3', 'Fcer1a', 'Prss34', 'Gata2'),
    'Endothelial cell' = c('Pecam1', 'Cdh5', 'Cldn5', 'Vwf'),
    'Fibroblast' = c('Col1a1', 'Col1a2', 'Dcn', 'Lum'),
    'Smooth muscle cell' = c('Tagln', 'Myh11', 'Acta2', 'Cnn1', 'Myl9'),
    'Pericyte' = c('Rgs5', 'Pdgfrb', 'Wif1', 'Chad', 'Cspg4')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE233625/GSE233625_pre_annt.csv"),row.names = FALSE, )


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    List of 14
 $ B cell               : chr [1:4] "Cd79a" "Cd79b" "Ms4a1" "Ly6d"
 $ T cell               : chr [1:3] "Cd3d" "Cd3g" "Trbc2"
 $ Natural killer cell  : chr [1:5] "Nkg7" "Klrb1c" "Gzma" "Gzmb" ...
 $ Dendritic cell       : chr [1:5] "Cd209a" "H2-Ab1" "H2-Eb1" "Itgax" ...
 $ Macrophage           : chr [1:5] "Cd68" "Adgre1" "C1qa" "C1qb" ...
 $ Monocyte             : chr [1:5] "Lyz2" "Cd14" "Ccr2" "Ly6c2" ...
 $ Mast cell            : chr [1:5] "Kit" "Cpa3" "Cma1" "Mcpt4" ...
 $ Erythrocyte/Erythroid: chr [1:6] "Hba-a1" "Hba-a2" "Hbb-bs" "Hbb-bt" ...
 $ Neutrophil           : chr [1:5] "S100a8" "S100a9" "Ly6g" "Cxcr2" ...
 $ Basophil             : chr [1:5] "Mcpt8" "Cd200r3" "Fcer1a" "Prss34" ...
 $ Endothelial cell     : chr [1:4] "Pecam1" "Cdh5" "Cldn5" "Vwf"
 $ Fibroblast           : c

Loading required package: SeuratObject
Loading required package: sp

Attaching package: ‘SeuratObject’

The following objects are masked from ‘package:base’:

    intersect, t


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



# GSE237067

In [10]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output/'
adata <- readRDS(file.path(outdir,"GSE237067/GSE237067_pre_annt.rds"))
level1_marker <- list(
    "B cell" = c('Cd79a', 'Cd79b', 'Ms4a1', 'Igkc', 'Cd22', 'Fcer2'),
    'T cell'= c('Cd2', 'Trac', 'Cd69', 'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Eomes', 'Lag3'),
    'Natural killer cell'= c('Nkg7', 'Xcl1', 'Ctsw', 'Xcl2', 'Cd160', 'Fcgr3', 'Prf1', 'Gnly'),

    'Dendritic cell'= c('Clec10a', 'Fcer1a', 'Cd1d1', 'H2-Ab1', 'H2-Eb1'),
    'Macrophage'= c('C1qa', 'C1qb', 'C1qc', 'Cd74', 'Cxcl8', 'Aif1', 'Cd68', 'Itgam', 'Csf1r', 'H2-Ab1', 'Lgals3', 'Cd163'),
    'Monocyte'= c('Fcn1', 'S100a8', 'S100a9', 'S100a12', 'Vcan', 'Cd52', 'Lyz2', 'Ctss', 'Cd14'),
    'Mast cell'= c('Tpsab1', 'Tpsb2', 'Hdc', 'Cma1'),
    'Erythrocyte/Erythroid'= c('Hbb-bs', 'Hba-a1', 'Hba-a2', 'Alas2', 'Ahsp', 'Slc4a1', 'Gypa', 'Klf1', 'Tmcc2'),
    'Neutrophil' = c('Nampt', 'Ifitm2', 'G0s2', 'Cxcl8', 'Neat1', 'Srgn', 'Aqp9', 'Sod2', 'Fcgr3', 'Ivns1abp'),
    'Basophil' = c('Tpsb2', 'Cpa3', 'Slc24a3', 'Fer', 'Kit', 'Hpgds', 'Sytl3', 'Maml3', 'Ell2', 'Ccac200', 'Akap13', 'Areg', 'Rhoh', 'Lrmda', 'Arid1b', 'Irak3', 'Tex14', 'Hpgd', 'Ercc1', 'Ctnnbl1', 'Zbtb20'),

    'Endothelial cell'= c('Pecam1', 'Vwf', 'Fabp4', 'Cldn5', 'Ifi27l2a', 'Ecscr', 'Dysf', 'Cd34', 'Col4a1', 'Col4a2', 'Sparcl1', 'Plvap', 'Mpzl2', 'Sulf1', 'Edn1'),

    'Fibroblast'= c('Lum', 'Dcn', 'Col1a1', 'Col1a2', 'Fbln1', 'Thy1'),
    'Smooth muscle cell'= c('Acta2', 'Myh11', 'Myl9', 'Tpm2', 'Cald1', 'Tagln', 'Tnfrsf11b', 'Lum', 'Apoe', 'Apoc1', 'Agt', 'Notch3', 'Pdgfrb', 'Mfap4'),

    'Pericyte'= c('Tagln', 'Lpp', 'Cald1', 'Tpm2', 'Myl9', 'Acta2', 'Map1b', 'Prkg1', 'Igfbp5', 'Synpo2', 'Eps8', 'Timp3', 'Lmod1', 'C11orf96', 'Inpp4b', 'Notch3', 'Ebf1', 'Steap4', 'mt-Rnr1', 'Crispld2', 'Sox5', 'Ppp1r14a', 'Filip1l', 'Lhfpl6', 'Ptprg')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE237067/GSE237067_pre_annt.csv"),row.names = FALSE, )


List of 14
 $ B cell               : chr [1:6] "Cd79a" "Cd79b" "Ms4a1" "Igkc" ...
 $ T cell               : chr [1:10] "Cd2" "Trac" "Cd69" "Cd3d" ...
 $ Natural killer cell  : chr [1:8] "Nkg7" "Xcl1" "Ctsw" "Xcl2" ...
 $ Dendritic cell       : chr [1:5] "Clec10a" "Fcer1a" "Cd1d1" "H2-Ab1" ...
 $ Macrophage           : chr [1:12] "C1qa" "C1qb" "C1qc" "Cd74" ...
 $ Monocyte             : chr [1:9] "Fcn1" "S100a8" "S100a9" "S100a12" ...
 $ Mast cell            : chr [1:4] "Tpsab1" "Tpsb2" "Hdc" "Cma1"
 $ Erythrocyte/Erythroid: chr [1:9] "Hbb-bs" "Hba-a1" "Hba-a2" "Alas2" ...
 $ Neutrophil           : chr [1:10] "Nampt" "Ifitm2" "G0s2" "Cxcl8" ...
 $ Basophil             : chr [1:21] "Tpsb2" "Cpa3" "Slc24a3" "Fer" ...
 $ Endothelial cell     : chr [1:15] "Pecam1" "Vwf" "Fabp4" "Cldn5" ...
 $ Fibroblast           : chr [1:6] "Lum" "Dcn" "Col1a1" "Col1a2" ...
 $ Smooth muscle cell   : chr [1:14] "Acta2" "Myh11" "Myl9" "Tpm2" ...
 $ Pericyte             : chr [1:25] "Tagln" "Lpp" "Cald1" "Tpm

## new_marker

我直接把rds复制到这个文件夹了

In [7]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output_new_marker/'
adata <- readRDS(file.path(outdir,"GSE237067/GSE237067_pre_annt.rds"))
level1_marker <- list(
    'B cell' = c('Cd79a', 'Cd79b', 'Ms4a1', 'Ly6d'),
    'T cell' = c('Cd3d', 'Cd3g', 'Trbc2'),
    'Natural killer cell' = c('Nkg7', 'Klrb1c', 'Gzma', 'Gzmb', 'Prf1'),
    'Dendritic cell' = c('Cd209a', 'H2-Ab1', 'H2-Eb1', 'Itgax', 'Flt3'),
    'Macrophage' = c('Cd68', 'Adgre1', 'C1qa', 'C1qb', 'Lyz2'),
    'Monocyte' = c('Lyz2', 'Cd14', 'Ccr2', 'Ly6c2', 'Cx3cr1'),
    'Mast cell' = c('Kit', 'Cpa3', 'Cma1', 'Mcpt4', 'Tpsb2'),
    'Erythrocyte/Erythroid' = c('Hba-a1', 'Hba-a2', 'Hbb-bs', 'Hbb-bt', 'Alas2', 'Klf1'),
    'Neutrophil' = c('S100a8', 'S100a9', 'Ly6g', 'Cxcr2', 'Retnlg'),
    'Basophil' = c('Mcpt8', 'Cd200r3', 'Fcer1a', 'Prss34', 'Gata2'),
    'Endothelial cell' = c('Pecam1', 'Cdh5', 'Cldn5', 'Vwf'),
    'Fibroblast' = c('Col1a1', 'Col1a2', 'Dcn', 'Lum'),
    'Smooth muscle cell' = c('Tagln', 'Myh11', 'Acta2', 'Cnn1', 'Myl9'),
    'Pericyte' = c('Rgs5', 'Pdgfrb', 'Wif1', 'Chad', 'Cspg4')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE237067/GSE237067_pre_annt.csv"),row.names = FALSE, )


List of 14
 $ B cell               : chr [1:4] "Cd79a" "Cd79b" "Ms4a1" "Ly6d"
 $ T cell               : chr [1:3] "Cd3d" "Cd3g" "Trbc2"
 $ Natural killer cell  : chr [1:5] "Nkg7" "Klrb1c" "Gzma" "Gzmb" ...
 $ Dendritic cell       : chr [1:5] "Cd209a" "H2-Ab1" "H2-Eb1" "Itgax" ...
 $ Macrophage           : chr [1:5] "Cd68" "Adgre1" "C1qa" "C1qb" ...
 $ Monocyte             : chr [1:5] "Lyz2" "Cd14" "Ccr2" "Ly6c2" ...
 $ Mast cell            : chr [1:5] "Kit" "Cpa3" "Cma1" "Mcpt4" ...
 $ Erythrocyte/Erythroid: chr [1:6] "Hba-a1" "Hba-a2" "Hbb-bs" "Hbb-bt" ...
 $ Neutrophil           : chr [1:5] "S100a8" "S100a9" "Ly6g" "Cxcr2" ...
 $ Basophil             : chr [1:5] "Mcpt8" "Cd200r3" "Fcer1a" "Prss34" ...
 $ Endothelial cell     : chr [1:4] "Pecam1" "Cdh5" "Cldn5" "Vwf"
 $ Fibroblast           : chr [1:4] "Col1a1" "Col1a2" "Dcn" "Lum"
 $ Smooth muscle cell   : chr [1:5] "Tagln" "Myh11" "Acta2" "Cnn1" ...
 $ Pericyte             : chr [1:5] "Rgs5" "Pdgfrb" "Wif1" "Chad" ...


# GSE269449

In [11]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output/'
adata <- readRDS(file.path(outdir,"GSE269449/GSE269449_pre_annt.rds"))
level1_marker <- list(
    "B cell" = c('Cd79a', 'Cd79b', 'Ms4a1', 'Igkc', 'Cd22', 'Fcer2'),
    'T cell'= c('Cd2', 'Trac', 'Cd69', 'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Eomes', 'Lag3'),
    'Natural killer cell'= c('Nkg7', 'Xcl1', 'Ctsw', 'Xcl2', 'Cd160', 'Fcgr3', 'Prf1', 'Gnly'),

    'Dendritic cell'= c('Clec10a', 'Fcer1a', 'Cd1d1', 'H2-Ab1', 'H2-Eb1'),
    'Macrophage'= c('C1qa', 'C1qb', 'C1qc', 'Cd74', 'Cxcl8', 'Aif1', 'Cd68', 'Itgam', 'Csf1r', 'H2-Ab1', 'Lgals3', 'Cd163'),
    'Monocyte'= c('Fcn1', 'S100a8', 'S100a9', 'S100a12', 'Vcan', 'Cd52', 'Lyz2', 'Ctss', 'Cd14'),
    'Mast cell'= c('Tpsab1', 'Tpsb2', 'Hdc', 'Cma1'),
    'Erythrocyte/Erythroid'= c('Hbb-bs', 'Hba-a1', 'Hba-a2', 'Alas2', 'Ahsp', 'Slc4a1', 'Gypa', 'Klf1', 'Tmcc2'),
    'Neutrophil' = c('Nampt', 'Ifitm2', 'G0s2', 'Cxcl8', 'Neat1', 'Srgn', 'Aqp9', 'Sod2', 'Fcgr3', 'Ivns1abp'),
    'Basophil' = c('Tpsb2', 'Cpa3', 'Slc24a3', 'Fer', 'Kit', 'Hpgds', 'Sytl3', 'Maml3', 'Ell2', 'Ccac200', 'Akap13', 'Areg', 'Rhoh', 'Lrmda', 'Arid1b', 'Irak3', 'Tex14', 'Hpgd', 'Ercc1', 'Ctnnbl1', 'Zbtb20'),

    'Endothelial cell'= c('Pecam1', 'Vwf', 'Fabp4', 'Cldn5', 'Ifi27l2a', 'Ecscr', 'Dysf', 'Cd34', 'Col4a1', 'Col4a2', 'Sparcl1', 'Plvap', 'Mpzl2', 'Sulf1', 'Edn1'),

    'Fibroblast'= c('Lum', 'Dcn', 'Col1a1', 'Col1a2', 'Fbln1', 'Thy1'),
    'Smooth muscle cell'= c('Acta2', 'Myh11', 'Myl9', 'Tpm2', 'Cald1', 'Tagln', 'Tnfrsf11b', 'Lum', 'Apoe', 'Apoc1', 'Agt', 'Notch3', 'Pdgfrb', 'Mfap4'),

    'Pericyte'= c('Tagln', 'Lpp', 'Cald1', 'Tpm2', 'Myl9', 'Acta2', 'Map1b', 'Prkg1', 'Igfbp5', 'Synpo2', 'Eps8', 'Timp3', 'Lmod1', 'C11orf96', 'Inpp4b', 'Notch3', 'Ebf1', 'Steap4', 'mt-Rnr1', 'Crispld2', 'Sox5', 'Ppp1r14a', 'Filip1l', 'Lhfpl6', 'Ptprg')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE269449/GSE269449_pre_annt.csv"),row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "Cd79a" "Cd79b" "Ms4a1" "Igkc" ...
 $ T cell               : chr [1:10] "Cd2" "Trac" "Cd69" "Cd3d" ...
 $ Natural killer cell  : chr [1:8] "Nkg7" "Xcl1" "Ctsw" "Xcl2" ...
 $ Dendritic cell       : chr [1:5] "Clec10a" "Fcer1a" "Cd1d1" "H2-Ab1" ...
 $ Macrophage           : chr [1:12] "C1qa" "C1qb" "C1qc" "Cd74" ...
 $ Monocyte             : chr [1:9] "Fcn1" "S100a8" "S100a9" "S100a12" ...
 $ Mast cell            : chr [1:4] "Tpsab1" "Tpsb2" "Hdc" "Cma1"
 $ Erythrocyte/Erythroid: chr [1:9] "Hbb-bs" "Hba-a1" "Hba-a2" "Alas2" ...
 $ Neutrophil           : chr [1:10] "Nampt" "Ifitm2" "G0s2" "Cxcl8" ...
 $ Basophil             : chr [1:21] "Tpsb2" "Cpa3" "Slc24a3" "Fer" ...
 $ Endothelial cell     : chr [1:15] "Pecam1" "Vwf" "Fabp4" "Cldn5" ...
 $ Fibroblast           : chr [1:6] "Lum" "Dcn" "Col1a1" "Col1a2" ...
 $ Smooth muscle cell   : chr [1:14] "Acta2" "Myh11" "Myl9" "Tpm2" ...
 $ Pericyte             : chr [1:25] "Tagln" "Lpp" "Cald1" "Tpm

## new_marker

In [8]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output_new_marker/'
adata <- readRDS(file.path(outdir,"GSE269449/GSE269449_pre_annt.rds"))
level1_marker <- list(
    'B cell' = c('Cd79a', 'Cd79b', 'Ms4a1', 'Ly6d'),
    'T cell' = c('Cd3d', 'Cd3g', 'Trbc2'),
    'Natural killer cell' = c('Nkg7', 'Klrb1c', 'Gzma', 'Gzmb', 'Prf1'),
    'Dendritic cell' = c('Cd209a', 'H2-Ab1', 'H2-Eb1', 'Itgax', 'Flt3'),
    'Macrophage' = c('Cd68', 'Adgre1', 'C1qa', 'C1qb', 'Lyz2'),
    'Monocyte' = c('Lyz2', 'Cd14', 'Ccr2', 'Ly6c2', 'Cx3cr1'),
    'Mast cell' = c('Kit', 'Cpa3', 'Cma1', 'Mcpt4', 'Tpsb2'),
    'Erythrocyte/Erythroid' = c('Hba-a1', 'Hba-a2', 'Hbb-bs', 'Hbb-bt', 'Alas2', 'Klf1'),
    'Neutrophil' = c('S100a8', 'S100a9', 'Ly6g', 'Cxcr2', 'Retnlg'),
    'Basophil' = c('Mcpt8', 'Cd200r3', 'Fcer1a', 'Prss34', 'Gata2'),
    'Endothelial cell' = c('Pecam1', 'Cdh5', 'Cldn5', 'Vwf'),
    'Fibroblast' = c('Col1a1', 'Col1a2', 'Dcn', 'Lum'),
    'Smooth muscle cell' = c('Tagln', 'Myh11', 'Acta2', 'Cnn1', 'Myl9'),
    'Pericyte' = c('Rgs5', 'Pdgfrb', 'Wif1', 'Chad', 'Cspg4')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE269449/GSE269449_pre_annt.csv"),row.names = FALSE, )

List of 14
 $ B cell               : chr [1:4] "Cd79a" "Cd79b" "Ms4a1" "Ly6d"
 $ T cell               : chr [1:3] "Cd3d" "Cd3g" "Trbc2"
 $ Natural killer cell  : chr [1:5] "Nkg7" "Klrb1c" "Gzma" "Gzmb" ...
 $ Dendritic cell       : chr [1:5] "Cd209a" "H2-Ab1" "H2-Eb1" "Itgax" ...
 $ Macrophage           : chr [1:5] "Cd68" "Adgre1" "C1qa" "C1qb" ...
 $ Monocyte             : chr [1:5] "Lyz2" "Cd14" "Ccr2" "Ly6c2" ...
 $ Mast cell            : chr [1:5] "Kit" "Cpa3" "Cma1" "Mcpt4" ...
 $ Erythrocyte/Erythroid: chr [1:6] "Hba-a1" "Hba-a2" "Hbb-bs" "Hbb-bt" ...
 $ Neutrophil           : chr [1:5] "S100a8" "S100a9" "Ly6g" "Cxcr2" ...
 $ Basophil             : chr [1:5] "Mcpt8" "Cd200r3" "Fcer1a" "Prss34" ...
 $ Endothelial cell     : chr [1:4] "Pecam1" "Cdh5" "Cldn5" "Vwf"
 $ Fibroblast           : chr [1:4] "Col1a1" "Col1a2" "Dcn" "Lum"
 $ Smooth muscle cell   : chr [1:5] "Tagln" "Myh11" "Acta2" "Cnn1" ...
 $ Pericyte             : chr [1:5] "Rgs5" "Pdgfrb" "Wif1" "Chad" ...


# GSE284253

In [10]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output/'
adata <- readRDS(file.path(outdir,"GSE284253/GSE284253_pre_annt.rds"))
level1_marker <- list(
    "B cell" = c('Cd79a', 'Cd79b', 'Ms4a1', 'Igkc', 'Cd22', 'Fcer2'),
    'T cell'= c('Cd2', 'Trac', 'Cd69', 'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Eomes', 'Lag3'),
    'Natural killer cell'= c('Nkg7', 'Xcl1', 'Ctsw', 'Xcl2', 'Cd160', 'Fcgr3', 'Prf1', 'Gnly'),

    'Dendritic cell'= c('Clec10a', 'Fcer1a', 'Cd1d1', 'H2-Ab1', 'H2-Eb1'),
    'Macrophage'= c('C1qa', 'C1qb', 'C1qc', 'Cd74', 'Cxcl8', 'Aif1', 'Cd68', 'Itgam', 'Csf1r', 'H2-Ab1', 'Lgals3', 'Cd163'),
    'Monocyte'= c('Fcn1', 'S100a8', 'S100a9', 'S100a12', 'Vcan', 'Cd52', 'Lyz2', 'Ctss', 'Cd14'),
    'Mast cell'= c('Tpsab1', 'Tpsb2', 'Hdc', 'Cma1'),
    'Erythrocyte/Erythroid'= c('Hbb-bs', 'Hba-a1', 'Hba-a2', 'Alas2', 'Ahsp', 'Slc4a1', 'Gypa', 'Klf1', 'Tmcc2'),
    'Neutrophil' = c('Nampt', 'Ifitm2', 'G0s2', 'Cxcl8', 'Neat1', 'Srgn', 'Aqp9', 'Sod2', 'Fcgr3', 'Ivns1abp'),
    'Basophil' = c('Tpsb2', 'Cpa3', 'Slc24a3', 'Fer', 'Kit', 'Hpgds', 'Sytl3', 'Maml3', 'Ell2', 'Ccac200', 'Akap13', 'Areg', 'Rhoh', 'Lrmda', 'Arid1b', 'Irak3', 'Tex14', 'Hpgd', 'Ercc1', 'Ctnnbl1', 'Zbtb20'),

    'Endothelial cell'= c('Pecam1', 'Vwf', 'Fabp4', 'Cldn5', 'Ifi27l2a', 'Ecscr', 'Dysf', 'Cd34', 'Col4a1', 'Col4a2', 'Sparcl1', 'Plvap', 'Mpzl2', 'Sulf1', 'Edn1'),

    'Fibroblast'= c('Lum', 'Dcn', 'Col1a1', 'Col1a2', 'Fbln1', 'Thy1'),
    'Smooth muscle cell'= c('Acta2', 'Myh11', 'Myl9', 'Tpm2', 'Cald1', 'Tagln', 'Tnfrsf11b', 'Lum', 'Apoe', 'Apoc1', 'Agt', 'Notch3', 'Pdgfrb', 'Mfap4'),

    'Pericyte'= c('Tagln', 'Lpp', 'Cald1', 'Tpm2', 'Myl9', 'Acta2', 'Map1b', 'Prkg1', 'Igfbp5', 'Synpo2', 'Eps8', 'Timp3', 'Lmod1', 'C11orf96', 'Inpp4b', 'Notch3', 'Ebf1', 'Steap4', 'mt-Rnr1', 'Crispld2', 'Sox5', 'Ppp1r14a', 'Filip1l', 'Lhfpl6', 'Ptprg')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE284253/GSE284253_pre_annt.csv"),row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "Cd79a" "Cd79b" "Ms4a1" "Igkc" ...
 $ T cell               : chr [1:10] "Cd2" "Trac" "Cd69" "Cd3d" ...
 $ Natural killer cell  : chr [1:8] "Nkg7" "Xcl1" "Ctsw" "Xcl2" ...
 $ Dendritic cell       : chr [1:5] "Clec10a" "Fcer1a" "Cd1d1" "H2-Ab1" ...
 $ Macrophage           : chr [1:12] "C1qa" "C1qb" "C1qc" "Cd74" ...
 $ Monocyte             : chr [1:9] "Fcn1" "S100a8" "S100a9" "S100a12" ...
 $ Mast cell            : chr [1:4] "Tpsab1" "Tpsb2" "Hdc" "Cma1"
 $ Erythrocyte/Erythroid: chr [1:9] "Hbb-bs" "Hba-a1" "Hba-a2" "Alas2" ...
 $ Neutrophil           : chr [1:10] "Nampt" "Ifitm2" "G0s2" "Cxcl8" ...
 $ Basophil             : chr [1:21] "Tpsb2" "Cpa3" "Slc24a3" "Fer" ...
 $ Endothelial cell     : chr [1:15] "Pecam1" "Vwf" "Fabp4" "Cldn5" ...
 $ Fibroblast           : chr [1:6] "Lum" "Dcn" "Col1a1" "Col1a2" ...
 $ Smooth muscle cell   : chr [1:14] "Acta2" "Myh11" "Myl9" "Tpm2" ...
 $ Pericyte             : chr [1:25] "Tagln" "Lpp" "Cald1" "Tpm

## new_marker

In [11]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output_new_marker/'
adata <- readRDS(file.path(outdir,"GSE284253/GSE284253_pre_annt.rds"))
level1_marker <- list(
    'B cell' = c('Cd79a', 'Cd79b', 'Ms4a1', 'Ly6d'),
    'T cell' = c('Cd3d', 'Cd3g', 'Trbc2'),
    'Natural killer cell' = c('Nkg7', 'Klrb1c', 'Gzma', 'Gzmb', 'Prf1'),
    'Dendritic cell' = c('Cd209a', 'H2-Ab1', 'H2-Eb1', 'Itgax', 'Flt3'),
    'Macrophage' = c('Cd68', 'Adgre1', 'C1qa', 'C1qb', 'Lyz2'),
    'Monocyte' = c('Lyz2', 'Cd14', 'Ccr2', 'Ly6c2', 'Cx3cr1'),
    'Mast cell' = c('Kit', 'Cpa3', 'Cma1', 'Mcpt4', 'Tpsb2'),
    'Erythrocyte/Erythroid' = c('Hba-a1', 'Hba-a2', 'Hbb-bs', 'Hbb-bt', 'Alas2', 'Klf1'),
    'Neutrophil' = c('S100a8', 'S100a9', 'Ly6g', 'Cxcr2', 'Retnlg'),
    'Basophil' = c('Mcpt8', 'Cd200r3', 'Fcer1a', 'Prss34', 'Gata2'),
    'Endothelial cell' = c('Pecam1', 'Cdh5', 'Cldn5', 'Vwf'),
    'Fibroblast' = c('Col1a1', 'Col1a2', 'Dcn', 'Lum'),
    'Smooth muscle cell' = c('Tagln', 'Myh11', 'Acta2', 'Cnn1', 'Myl9'),
    'Pericyte' = c('Rgs5', 'Pdgfrb', 'Wif1', 'Chad', 'Cspg4')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE284253/GSE284253_pre_annt.csv"),row.names = FALSE, )

List of 14
 $ B cell               : chr [1:4] "Cd79a" "Cd79b" "Ms4a1" "Ly6d"
 $ T cell               : chr [1:3] "Cd3d" "Cd3g" "Trbc2"
 $ Natural killer cell  : chr [1:5] "Nkg7" "Klrb1c" "Gzma" "Gzmb" ...
 $ Dendritic cell       : chr [1:5] "Cd209a" "H2-Ab1" "H2-Eb1" "Itgax" ...
 $ Macrophage           : chr [1:5] "Cd68" "Adgre1" "C1qa" "C1qb" ...
 $ Monocyte             : chr [1:5] "Lyz2" "Cd14" "Ccr2" "Ly6c2" ...
 $ Mast cell            : chr [1:5] "Kit" "Cpa3" "Cma1" "Mcpt4" ...
 $ Erythrocyte/Erythroid: chr [1:6] "Hba-a1" "Hba-a2" "Hbb-bs" "Hbb-bt" ...
 $ Neutrophil           : chr [1:5] "S100a8" "S100a9" "Ly6g" "Cxcr2" ...
 $ Basophil             : chr [1:5] "Mcpt8" "Cd200r3" "Fcer1a" "Prss34" ...
 $ Endothelial cell     : chr [1:4] "Pecam1" "Cdh5" "Cldn5" "Vwf"
 $ Fibroblast           : chr [1:4] "Col1a1" "Col1a2" "Dcn" "Lum"
 $ Smooth muscle cell   : chr [1:5] "Tagln" "Myh11" "Acta2" "Cnn1" ...
 $ Pericyte             : chr [1:5] "Rgs5" "Pdgfrb" "Wif1" "Chad" ...


# GSE279601

In [13]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output/'
adata <- readRDS(file.path(outdir,"GSE279601/GSE279601_pre_annt.rds"))
level1_marker <- list(
    "B cell" = c('Cd79a', 'Cd79b', 'Ms4a1', 'Igkc', 'Cd22', 'Fcer2'),
    'T cell'= c('Cd2', 'Trac', 'Cd69', 'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Eomes', 'Lag3'),
    'Natural killer cell'= c('Nkg7', 'Xcl1', 'Ctsw', 'Xcl2', 'Cd160', 'Fcgr3', 'Prf1', 'Gnly'),

    'Dendritic cell'= c('Clec10a', 'Fcer1a', 'Cd1d1', 'H2-Ab1', 'H2-Eb1'),
    'Macrophage'= c('C1qa', 'C1qb', 'C1qc', 'Cd74', 'Cxcl8', 'Aif1', 'Cd68', 'Itgam', 'Csf1r', 'H2-Ab1', 'Lgals3', 'Cd163'),
    'Monocyte'= c('Fcn1', 'S100a8', 'S100a9', 'S100a12', 'Vcan', 'Cd52', 'Lyz2', 'Ctss', 'Cd14'),
    'Mast cell'= c('Tpsab1', 'Tpsb2', 'Hdc', 'Cma1'),
    'Erythrocyte/Erythroid'= c('Hbb-bs', 'Hba-a1', 'Hba-a2', 'Alas2', 'Ahsp', 'Slc4a1', 'Gypa', 'Klf1', 'Tmcc2'),
    'Neutrophil' = c('Nampt', 'Ifitm2', 'G0s2', 'Cxcl8', 'Neat1', 'Srgn', 'Aqp9', 'Sod2', 'Fcgr3', 'Ivns1abp'),
    'Basophil' = c('Tpsb2', 'Cpa3', 'Slc24a3', 'Fer', 'Kit', 'Hpgds', 'Sytl3', 'Maml3', 'Ell2', 'Ccac200', 'Akap13', 'Areg', 'Rhoh', 'Lrmda', 'Arid1b', 'Irak3', 'Tex14', 'Hpgd', 'Ercc1', 'Ctnnbl1', 'Zbtb20'),

    'Endothelial cell'= c('Pecam1', 'Vwf', 'Fabp4', 'Cldn5', 'Ifi27l2a', 'Ecscr', 'Dysf', 'Cd34', 'Col4a1', 'Col4a2', 'Sparcl1', 'Plvap', 'Mpzl2', 'Sulf1', 'Edn1'),

    'Fibroblast'= c('Lum', 'Dcn', 'Col1a1', 'Col1a2', 'Fbln1', 'Thy1'),
    'Smooth muscle cell'= c('Acta2', 'Myh11', 'Myl9', 'Tpm2', 'Cald1', 'Tagln', 'Tnfrsf11b', 'Lum', 'Apoe', 'Apoc1', 'Agt', 'Notch3', 'Pdgfrb', 'Mfap4'),

    'Pericyte'= c('Tagln', 'Lpp', 'Cald1', 'Tpm2', 'Myl9', 'Acta2', 'Map1b', 'Prkg1', 'Igfbp5', 'Synpo2', 'Eps8', 'Timp3', 'Lmod1', 'C11orf96', 'Inpp4b', 'Notch3', 'Ebf1', 'Steap4', 'mt-Rnr1', 'Crispld2', 'Sox5', 'Ppp1r14a', 'Filip1l', 'Lhfpl6', 'Ptprg')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE279601/GSE279601_pre_annt.csv"),row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "Cd79a" "Cd79b" "Ms4a1" "Igkc" ...
 $ T cell               : chr [1:10] "Cd2" "Trac" "Cd69" "Cd3d" ...
 $ Natural killer cell  : chr [1:8] "Nkg7" "Xcl1" "Ctsw" "Xcl2" ...
 $ Dendritic cell       : chr [1:5] "Clec10a" "Fcer1a" "Cd1d1" "H2-Ab1" ...
 $ Macrophage           : chr [1:12] "C1qa" "C1qb" "C1qc" "Cd74" ...
 $ Monocyte             : chr [1:9] "Fcn1" "S100a8" "S100a9" "S100a12" ...
 $ Mast cell            : chr [1:4] "Tpsab1" "Tpsb2" "Hdc" "Cma1"
 $ Erythrocyte/Erythroid: chr [1:9] "Hbb-bs" "Hba-a1" "Hba-a2" "Alas2" ...
 $ Neutrophil           : chr [1:10] "Nampt" "Ifitm2" "G0s2" "Cxcl8" ...
 $ Basophil             : chr [1:21] "Tpsb2" "Cpa3" "Slc24a3" "Fer" ...
 $ Endothelial cell     : chr [1:15] "Pecam1" "Vwf" "Fabp4" "Cldn5" ...
 $ Fibroblast           : chr [1:6] "Lum" "Dcn" "Col1a1" "Col1a2" ...
 $ Smooth muscle cell   : chr [1:14] "Acta2" "Myh11" "Myl9" "Tpm2" ...
 $ Pericyte             : chr [1:25] "Tagln" "Lpp" "Cald1" "Tpm

## new_marker

In [12]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output_new_marker/'
adata <- readRDS(file.path(outdir,"GSE279601/GSE279601_pre_annt.rds"))
level1_marker <- list(
    'B cell' = c('Cd79a', 'Cd79b', 'Ms4a1', 'Ly6d'),
    'T cell' = c('Cd3d', 'Cd3g', 'Trbc2'),
    'Natural killer cell' = c('Nkg7', 'Klrb1c', 'Gzma', 'Gzmb', 'Prf1'),
    'Dendritic cell' = c('Cd209a', 'H2-Ab1', 'H2-Eb1', 'Itgax', 'Flt3'),
    'Macrophage' = c('Cd68', 'Adgre1', 'C1qa', 'C1qb', 'Lyz2'),
    'Monocyte' = c('Lyz2', 'Cd14', 'Ccr2', 'Ly6c2', 'Cx3cr1'),
    'Mast cell' = c('Kit', 'Cpa3', 'Cma1', 'Mcpt4', 'Tpsb2'),
    'Erythrocyte/Erythroid' = c('Hba-a1', 'Hba-a2', 'Hbb-bs', 'Hbb-bt', 'Alas2', 'Klf1'),
    'Neutrophil' = c('S100a8', 'S100a9', 'Ly6g', 'Cxcr2', 'Retnlg'),
    'Basophil' = c('Mcpt8', 'Cd200r3', 'Fcer1a', 'Prss34', 'Gata2'),
    'Endothelial cell' = c('Pecam1', 'Cdh5', 'Cldn5', 'Vwf'),
    'Fibroblast' = c('Col1a1', 'Col1a2', 'Dcn', 'Lum'),
    'Smooth muscle cell' = c('Tagln', 'Myh11', 'Acta2', 'Cnn1', 'Myl9'),
    'Pericyte' = c('Rgs5', 'Pdgfrb', 'Wif1', 'Chad', 'Cspg4')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE279601/GSE279601_pre_annt.csv"),row.names = FALSE, )

List of 14
 $ B cell               : chr [1:4] "Cd79a" "Cd79b" "Ms4a1" "Ly6d"
 $ T cell               : chr [1:3] "Cd3d" "Cd3g" "Trbc2"
 $ Natural killer cell  : chr [1:5] "Nkg7" "Klrb1c" "Gzma" "Gzmb" ...
 $ Dendritic cell       : chr [1:5] "Cd209a" "H2-Ab1" "H2-Eb1" "Itgax" ...
 $ Macrophage           : chr [1:5] "Cd68" "Adgre1" "C1qa" "C1qb" ...
 $ Monocyte             : chr [1:5] "Lyz2" "Cd14" "Ccr2" "Ly6c2" ...
 $ Mast cell            : chr [1:5] "Kit" "Cpa3" "Cma1" "Mcpt4" ...
 $ Erythrocyte/Erythroid: chr [1:6] "Hba-a1" "Hba-a2" "Hbb-bs" "Hbb-bt" ...
 $ Neutrophil           : chr [1:5] "S100a8" "S100a9" "Ly6g" "Cxcr2" ...
 $ Basophil             : chr [1:5] "Mcpt8" "Cd200r3" "Fcer1a" "Prss34" ...
 $ Endothelial cell     : chr [1:4] "Pecam1" "Cdh5" "Cldn5" "Vwf"
 $ Fibroblast           : chr [1:4] "Col1a1" "Col1a2" "Dcn" "Lum"
 $ Smooth muscle cell   : chr [1:5] "Tagln" "Myh11" "Acta2" "Cnn1" ...
 $ Pericyte             : chr [1:5] "Rgs5" "Pdgfrb" "Wif1" "Chad" ...


# GSE155513##

In [14]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output/'
adata <- readRDS(file.path(outdir,"GSE155513/GSE155513_pre_annt.rds"))
level1_marker <- list(
    "B cell" = c('Cd79a', 'Cd79b', 'Ms4a1', 'Igkc', 'Cd22', 'Fcer2'),
    'T cell'= c('Cd2', 'Trac', 'Cd69', 'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Eomes', 'Lag3'),
    'Natural killer cell'= c('Nkg7', 'Xcl1', 'Ctsw', 'Xcl2', 'Cd160', 'Fcgr3', 'Prf1', 'Gnly'),

    'Dendritic cell'= c('Clec10a', 'Fcer1a', 'Cd1d1', 'H2-Ab1', 'H2-Eb1'),
    'Macrophage'= c('C1qa', 'C1qb', 'C1qc', 'Cd74', 'Cxcl8', 'Aif1', 'Cd68', 'Itgam', 'Csf1r', 'H2-Ab1', 'Lgals3', 'Cd163'),
    'Monocyte'= c('Fcn1', 'S100a8', 'S100a9', 'S100a12', 'Vcan', 'Cd52', 'Lyz2', 'Ctss', 'Cd14'),
    'Mast cell'= c('Tpsab1', 'Tpsb2', 'Hdc', 'Cma1'),
    'Erythrocyte/Erythroid'= c('Hbb-bs', 'Hba-a1', 'Hba-a2', 'Alas2', 'Ahsp', 'Slc4a1', 'Gypa', 'Klf1', 'Tmcc2'),
    'Neutrophil' = c('Nampt', 'Ifitm2', 'G0s2', 'Cxcl8', 'Neat1', 'Srgn', 'Aqp9', 'Sod2', 'Fcgr3', 'Ivns1abp'),
    'Basophil' = c('Tpsb2', 'Cpa3', 'Slc24a3', 'Fer', 'Kit', 'Hpgds', 'Sytl3', 'Maml3', 'Ell2', 'Ccac200', 'Akap13', 'Areg', 'Rhoh', 'Lrmda', 'Arid1b', 'Irak3', 'Tex14', 'Hpgd', 'Ercc1', 'Ctnnbl1', 'Zbtb20'),

    'Endothelial cell'= c('Pecam1', 'Vwf', 'Fabp4', 'Cldn5', 'Ifi27l2a', 'Ecscr', 'Dysf', 'Cd34', 'Col4a1', 'Col4a2', 'Sparcl1', 'Plvap', 'Mpzl2', 'Sulf1', 'Edn1'),

    'Fibroblast'= c('Lum', 'Dcn', 'Col1a1', 'Col1a2', 'Fbln1', 'Thy1'),
    'Smooth muscle cell'= c('Acta2', 'Myh11', 'Myl9', 'Tpm2', 'Cald1', 'Tagln', 'Tnfrsf11b', 'Lum', 'Apoe', 'Apoc1', 'Agt', 'Notch3', 'Pdgfrb', 'Mfap4'),

    'Pericyte'= c('Tagln', 'Lpp', 'Cald1', 'Tpm2', 'Myl9', 'Acta2', 'Map1b', 'Prkg1', 'Igfbp5', 'Synpo2', 'Eps8', 'Timp3', 'Lmod1', 'C11orf96', 'Inpp4b', 'Notch3', 'Ebf1', 'Steap4', 'mt-Rnr1', 'Crispld2', 'Sox5', 'Ppp1r14a', 'Filip1l', 'Lhfpl6', 'Ptprg')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE155513/GSE155513_pre_annt.csv"),row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "Cd79a" "Cd79b" "Ms4a1" "Igkc" ...
 $ T cell               : chr [1:10] "Cd2" "Trac" "Cd69" "Cd3d" ...
 $ Natural killer cell  : chr [1:8] "Nkg7" "Xcl1" "Ctsw" "Xcl2" ...
 $ Dendritic cell       : chr [1:5] "Clec10a" "Fcer1a" "Cd1d1" "H2-Ab1" ...
 $ Macrophage           : chr [1:12] "C1qa" "C1qb" "C1qc" "Cd74" ...
 $ Monocyte             : chr [1:9] "Fcn1" "S100a8" "S100a9" "S100a12" ...
 $ Mast cell            : chr [1:4] "Tpsab1" "Tpsb2" "Hdc" "Cma1"
 $ Erythrocyte/Erythroid: chr [1:9] "Hbb-bs" "Hba-a1" "Hba-a2" "Alas2" ...
 $ Neutrophil           : chr [1:10] "Nampt" "Ifitm2" "G0s2" "Cxcl8" ...
 $ Basophil             : chr [1:21] "Tpsb2" "Cpa3" "Slc24a3" "Fer" ...
 $ Endothelial cell     : chr [1:15] "Pecam1" "Vwf" "Fabp4" "Cldn5" ...
 $ Fibroblast           : chr [1:6] "Lum" "Dcn" "Col1a1" "Col1a2" ...
 $ Smooth muscle cell   : chr [1:14] "Acta2" "Myh11" "Myl9" "Tpm2" ...
 $ Pericyte             : chr [1:25] "Tagln" "Lpp" "Cald1" "Tpm

# GSE239591##

In [ ]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output/'
adata <- readRDS(file.path(outdir,"GSE239591/GSE239591_pre_annt.rds"))
level1_marker <- list(
    "B cell" = c('Cd79a', 'Cd79b', 'Ms4a1', 'Igkc', 'Cd22', 'Fcer2'),
    'T cell'= c('Cd2', 'Trac', 'Cd69', 'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Eomes', 'Lag3'),
    'Natural killer cell'= c('Nkg7', 'Xcl1', 'Ctsw', 'Xcl2', 'Cd160', 'Fcgr3', 'Prf1', 'Gnly'),

    'Dendritic cell'= c('Clec10a', 'Fcer1a', 'Cd1d1', 'H2-Ab1', 'H2-Eb1'),
    'Macrophage'= c('C1qa', 'C1qb', 'C1qc', 'Cd74', 'Cxcl8', 'Aif1', 'Cd68', 'Itgam', 'Csf1r', 'H2-Ab1', 'Lgals3', 'Cd163'),
    'Monocyte'= c('Fcn1', 'S100a8', 'S100a9', 'S100a12', 'Vcan', 'Cd52', 'Lyz2', 'Ctss', 'Cd14'),
    'Mast cell'= c('Tpsab1', 'Tpsb2', 'Hdc', 'Cma1'),
    'Erythrocyte/Erythroid'= c('Hbb-bs', 'Hba-a1', 'Hba-a2', 'Alas2', 'Ahsp', 'Slc4a1', 'Gypa', 'Klf1', 'Tmcc2'),
    'Neutrophil' = c('Nampt', 'Ifitm2', 'G0s2', 'Cxcl8', 'Neat1', 'Srgn', 'Aqp9', 'Sod2', 'Fcgr3', 'Ivns1abp'),
    'Basophil' = c('Tpsb2', 'Cpa3', 'Slc24a3', 'Fer', 'Kit', 'Hpgds', 'Sytl3', 'Maml3', 'Ell2', 'Ccac200', 'Akap13', 'Areg', 'Rhoh', 'Lrmda', 'Arid1b', 'Irak3', 'Tex14', 'Hpgd', 'Ercc1', 'Ctnnbl1', 'Zbtb20'),

    'Endothelial cell'= c('Pecam1', 'Vwf', 'Fabp4', 'Cldn5', 'Ifi27l2a', 'Ecscr', 'Dysf', 'Cd34', 'Col4a1', 'Col4a2', 'Sparcl1', 'Plvap', 'Mpzl2', 'Sulf1', 'Edn1'),

    'Fibroblast'= c('Lum', 'Dcn', 'Col1a1', 'Col1a2', 'Fbln1', 'Thy1'),
    'Smooth muscle cell'= c('Acta2', 'Myh11', 'Myl9', 'Tpm2', 'Cald1', 'Tagln', 'Tnfrsf11b', 'Lum', 'Apoe', 'Apoc1', 'Agt', 'Notch3', 'Pdgfrb', 'Mfap4'),

    'Pericyte'= c('Tagln', 'Lpp', 'Cald1', 'Tpm2', 'Myl9', 'Acta2', 'Map1b', 'Prkg1', 'Igfbp5', 'Synpo2', 'Eps8', 'Timp3', 'Lmod1', 'C11orf96', 'Inpp4b', 'Notch3', 'Ebf1', 'Steap4', 'mt-Rnr1', 'Crispld2', 'Sox5', 'Ppp1r14a', 'Filip1l', 'Lhfpl6', 'Ptprg')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = file.path(outdir,"GSE239591/GSE239591_pre_annt.csv"),row.names = FALSE, )